# Tumor Synthesis MVP — Colab Training Notebook
**3D Latent Diffusion for CT Tumor Synthesis**  
Team: Qi Chen, Emily Kim, Yoonseo Bae

## Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your `tumor-synthesis/` project folder to Google Drive
3. Upload pancreas data to `MyDrive/tumor-synthesis/data/raw/pancreas/images/` and `.../labels/`
4. Run cells top to bottom

## Pipeline
- **Stage 1**: Train 3D autoencoder on unlabeled CT patches
- **Stage 2**: Train conditional latent diffusion (tumor mask + healthy context)
- **Stage 3**: Synthesize tumors → train segmenter → evaluate

## 0. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q monai>=1.3.0 monai-generative>=0.2.3 nibabel einops

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = '/content/drive/MyDrive/tumor-synthesis'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Verify data is in place
import glob
images = glob.glob('data/raw/pancreas/images/*.nii.gz')
labels = glob.glob('data/raw/pancreas/labels/*.nii.gz')
print(f'Pancreas images: {len(images)}')
print(f'Pancreas labels: {len(labels)}')
assert len(images) > 0, 'No images found! Check data/raw/pancreas/images/'
assert len(labels) > 0, 'No labels found! Check data/raw/pancreas/labels/'

## 1. Imports & Config

In [ ]:
import yaml
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from torch.cuda.amp import GradScaler, autocast

from utils.config import get_device
from models.autoencoder import CTAutoencoder, kl_loss
from models.diffusion import ConditionalLDM
from models.segmentation import TumorSegmenter, get_loss, get_sliding_window_inferer
from data.data_module import get_autoencoder_loaders, get_diffusion_loaders, get_segmentation_loaders
from utils.visualization import compare_recon, show_ct_slices
from utils.metrics import dice_score, tumor_sensitivity, stratified_sensitivity

# Load colab config
with open('configs/colab.yaml') as f:
    CFG = yaml.safe_load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_SIZE = CFG['data']['patch_size']
RAW_DIR = CFG['data']['raw_dir']
CKPT_DIR = Path(CFG['output']['checkpoint_dir'])
SAMPLE_DIR = Path(CFG['output']['sample_dir'])
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(CFG['training']['seed'])
print(f'Device: {DEVICE} | Patch size: {PATCH_SIZE}')

---
## Stage 1 — Autoencoder Training
Train a 3D VAE-style autoencoder on **unlabeled** CT patches.  
Goal: learn a compact latent space that captures CT anatomy & texture.  
> **Source**: Proposal Stage 1 + MONAI AutoencoderKL

In [ ]:
# Build data loaders (unlabeled — uses all images)
ae_cfg = CFG['autoencoder']
train_loader, val_loader = get_autoencoder_loaders(
    raw_dir=RAW_DIR,
    organs=['pancreas'],
    patch_size=PATCH_SIZE,
    batch_size=ae_cfg['training']['batch_size'],
    num_workers=CFG['data']['num_workers'],
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# Build autoencoder
m = ae_cfg['model']
ae_model = CTAutoencoder(
    latent_channels=m['latent_channels'],
    num_channels=tuple(m['channels']),
    attention_levels=tuple(m['attention_levels']),
    num_res_blocks=m['num_res_blocks'],
    norm_num_groups=m['norm_num_groups'],
).to(DEVICE)

n_params = sum(p.numel() for p in ae_model.parameters())
print(f'Autoencoder parameters: {n_params:,}')

In [ ]:
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=ae_cfg['training']['lr'])
ae_scaler = GradScaler()
ae_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ae_optimizer,
    T_max=ae_cfg['training']['max_epochs'] - ae_cfg['training']['warmup_epochs'],
    eta_min=1e-6
)

KL_W = m['kl_weight']
MAX_EPOCHS = ae_cfg['training']['max_epochs']
VAL_INTERVAL = ae_cfg['training']['val_interval']
WARMUP = ae_cfg['training']['warmup_epochs']
best_val = float('inf')

ae_train_losses, ae_val_losses = [], []
print(f'Training for {MAX_EPOCHS} epochs...')

In [ ]:
# Resume from checkpoint if exists
ae_ckpt_path = CKPT_DIR / 'autoencoder_best.pth'
start_epoch = 1
if ae_ckpt_path.exists():
    ckpt = torch.load(ae_ckpt_path, map_location=DEVICE)
    ae_model.load_state_dict(ckpt['model'])
    ae_optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    best_val = ckpt.get('best_val', float('inf'))
    print(f'Resumed from epoch {ckpt["epoch"]} (val_loss={best_val:.4f})')
else:
    print('Starting fresh')

In [ ]:
for epoch in range(start_epoch, MAX_EPOCHS + 1):
    # --- Train ---
    ae_model.train()
    train_loss = 0.0
    for batch in train_loader:
        x = batch['image'].to(DEVICE)
        ae_optimizer.zero_grad()
        with autocast():
            recon, mu, sigma = ae_model(x)
            loss = F.l1_loss(recon, x) + KL_W * kl_loss(mu, sigma)
        ae_scaler.scale(loss).backward()
        ae_scaler.unscale_(ae_optimizer)
        torch.nn.utils.clip_grad_norm_(ae_model.parameters(), 1.0)
        ae_scaler.step(ae_optimizer)
        ae_scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    ae_train_losses.append(train_loss)
    if epoch > WARMUP:
        ae_scheduler.step()

    # --- Validate ---
    if epoch % VAL_INTERVAL == 0:
        ae_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                x = batch['image'].to(DEVICE)
                recon, mu, sigma = ae_model(x)
                val_loss += (F.l1_loss(recon, x) + KL_W * kl_loss(mu, sigma)).item()
                # Save one reconstruction sample
                if i == 0:
                    orig = x[0, 0].cpu().numpy()
                    rec = recon[0, 0].clamp(0, 1).cpu().numpy()
                    compare_recon(orig, rec,
                        save_path=SAMPLE_DIR / f'ae_recon_epoch{epoch:04d}.png')
        val_loss /= len(val_loader)
        ae_val_losses.append((epoch, val_loss))

        if val_loss < best_val:
            best_val = val_loss
            torch.save({'model': ae_model.state_dict(),
                        'optimizer': ae_optimizer.state_dict(),
                        'epoch': epoch, 'best_val': best_val},
                       ae_ckpt_path)
        print(f'Epoch {epoch:03d}/{MAX_EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}'
              + (' [saved]' if val_loss == best_val else ''))
    else:
        if epoch % 10 == 0:
            print(f'Epoch {epoch:03d}/{MAX_EPOCHS} | train={train_loss:.4f}')

print(f'\nStage 1 done. Best val loss: {best_val:.4f}')

In [ ]:
# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(ae_train_losses, label='train')
if ae_val_losses:
    epochs_v, vals_v = zip(*ae_val_losses)
    plt.plot(epochs_v, vals_v, 'o-', label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss (L1 + KL)')
plt.title('Stage 1: Autoencoder Loss'); plt.legend(); plt.grid(True)
plt.savefig(SAMPLE_DIR / 'ae_loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize latest reconstruction
from PIL import Image
recon_imgs = sorted((SAMPLE_DIR).glob('ae_recon_epoch*.png'))
if recon_imgs:
    img = Image.open(recon_imgs[-1])
    plt.figure(figsize=(14, 4))
    plt.imshow(img); plt.axis('off')
    plt.title(f'Latest reconstruction: {recon_imgs[-1].name}')
    plt.show()

---
## Stage 2 — Conditional Latent Diffusion Training
Train a diffusion model conditioned on **tumor mask + healthy context**.  
Requires: Stage 1 checkpoint must exist.
> **Source**: Proposal Stage 2 + Rombach et al. LDM [3] + MONAI DiffusionModelUNet

In [ ]:
# Load frozen autoencoder
assert ae_ckpt_path.exists(), 'Run Stage 1 first!'
ae_model.eval()
for p in ae_model.parameters():
    p.requires_grad_(False)
print('Autoencoder loaded and frozen.')

# Labeled data only (has tumor masks)
diff_cfg = CFG['diffusion']
diff_train_loader, diff_val_loader = get_diffusion_loaders(
    raw_dir=RAW_DIR,
    organs=['pancreas'],
    patch_size=PATCH_SIZE,
    batch_size=diff_cfg['training']['batch_size'],
    num_workers=CFG['data']['num_workers'],
    num_samples_per_volume=2,
)
print(f'Diffusion train batches: {len(diff_train_loader)} | val: {len(diff_val_loader)}')

In [ ]:
dm = diff_cfg['model']
ds = diff_cfg['scheduler']
diff_model = ConditionalLDM(
    latent_channels=dm['out_channels'],
    num_channels=tuple(dm['num_channels']),
    attention_levels=tuple(dm['attention_levels']),
    num_res_blocks=dm['num_res_blocks'],
    num_head_channels=dm['num_head_channels'],
    norm_num_groups=dm['norm_num_groups'],
    num_train_timesteps=ds['num_train_timesteps'],
    beta_schedule=ds['beta_schedule'],
    beta_start=ds['beta_start'],
    beta_end=ds['beta_end'],
).to(DEVICE)
print(f'Diffusion parameters: {sum(p.numel() for p in diff_model.parameters()):,}')

In [ ]:
diff_optimizer = torch.optim.Adam(diff_model.parameters(), lr=diff_cfg['training']['lr'])
diff_scaler = GradScaler()
DIFF_MAX_EPOCHS = diff_cfg['training']['max_epochs']
DIFF_WARMUP = diff_cfg['training']['warmup_epochs']
DIFF_VAL_INTERVAL = diff_cfg['training']['val_interval']
diff_scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(
    diff_optimizer, T_max=DIFF_MAX_EPOCHS - DIFF_WARMUP, eta_min=1e-6
)

diff_ckpt_path = CKPT_DIR / 'diffusion_best.pth'
diff_start_epoch = 1
diff_best_val = float('inf')

if diff_ckpt_path.exists():
    ckpt = torch.load(diff_ckpt_path, map_location=DEVICE)
    diff_model.load_state_dict(ckpt['model'])
    diff_start_epoch = ckpt['epoch'] + 1
    diff_best_val = ckpt.get('best_val', float('inf'))
    print(f'Resumed diffusion from epoch {ckpt["epoch"]}')
else:
    print('Starting diffusion training fresh')

In [ ]:
diff_train_losses, diff_val_losses = [], []

for epoch in range(diff_start_epoch, DIFF_MAX_EPOCHS + 1):
    diff_model.train()
    train_loss = 0.0

    for batch in diff_train_loader:
        images = batch['image'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        # Build healthy context: zero out tumor region [PROPOSAL Stage 2]
        mask = (labels > 0).float()
        healthy = images * (1.0 - mask)

        with torch.no_grad():
            z_tumor = ae_model.encode_deterministic(images)
            z_context = ae_model.encode_deterministic(healthy)
        mask_ds = F.interpolate(mask, size=z_tumor.shape[2:], mode='nearest')

        diff_optimizer.zero_grad()
        with autocast():
            noise_pred, noise = diff_model(z_tumor, mask_ds, z_context)
            loss = F.mse_loss(noise_pred, noise)

        diff_scaler.scale(loss).backward()
        diff_scaler.unscale_(diff_optimizer)
        torch.nn.utils.clip_grad_norm_(diff_model.parameters(), 1.0)
        diff_scaler.step(diff_optimizer)
        diff_scaler.update()
        train_loss += loss.item()

    train_loss /= len(diff_train_loader)
    diff_train_losses.append(train_loss)
    if epoch > DIFF_WARMUP:
        diff_scheduler_lr.step()

    if epoch % DIFF_VAL_INTERVAL == 0:
        diff_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in diff_val_loader:
                images = batch['image'].to(DEVICE)
                labels = batch['label'].to(DEVICE)
                mask = (labels > 0).float()
                healthy = images * (1.0 - mask)
                z_tumor = ae_model.encode_deterministic(images)
                z_context = ae_model.encode_deterministic(healthy)
                mask_ds = F.interpolate(mask, size=z_tumor.shape[2:], mode='nearest')
                np_, n = diff_model(z_tumor, mask_ds, z_context)
                val_loss += F.mse_loss(np_, n).item()
        val_loss /= len(diff_val_loader)
        diff_val_losses.append((epoch, val_loss))

        if val_loss < diff_best_val:
            diff_best_val = val_loss
            torch.save({'model': diff_model.state_dict(),
                        'optimizer': diff_optimizer.state_dict(),
                        'epoch': epoch, 'best_val': diff_best_val},
                       diff_ckpt_path)
        print(f'Epoch {epoch:03d}/{DIFF_MAX_EPOCHS} | train={train_loss:.6f} | val={val_loss:.6f}'
              + (' [saved]' if val_loss == diff_best_val else ''))
    else:
        if epoch % 25 == 0:
            print(f'Epoch {epoch:03d}/{DIFF_MAX_EPOCHS} | train={train_loss:.6f}')

print(f'\nStage 2 done. Best val loss: {diff_best_val:.6f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(diff_train_losses, label='train')
if diff_val_losses:
    ev, vv = zip(*diff_val_losses)
    plt.plot(ev, vv, 'o-', label='val')
plt.xlabel('Epoch'); plt.ylabel('MSE loss')
plt.title('Stage 2: Diffusion Loss'); plt.legend(); plt.grid(True)
plt.savefig(SAMPLE_DIR / 'diff_loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Stage 3a — Tumor Synthesis
Generate synthetic (CT volume, tumor mask) pairs using trained models.
> **Source**: Proposal Stage 3 + Hu et al. mask generation [HU]

In [ ]:
import random
import nibabel as nib
from data.mask_generator import generate_tumor_mask
from utils.nifti_io import load_nifti, save_nifti, window_ct
from data.transforms import CT_WIN_MIN, CT_WIN_MAX

assert diff_ckpt_path.exists(), 'Run Stage 2 first!'
diff_model.eval()
ae_model.eval()

# Output dirs
synth_dir = SAMPLE_DIR / 'pancreas'
(synth_dir / 'images').mkdir(parents=True, exist_ok=True)
(synth_dir / 'labels').mkdir(parents=True, exist_ok=True)
(synth_dir / 'previews').mkdir(parents=True, exist_ok=True)

N_SYNTH = 50       # generate 50 synthetic pairs
N_INFER_STEPS = 20 # [PROPOSAL challenge 3] fewer steps
healthy_paths = sorted(Path(RAW_DIR + '/pancreas/images').glob('*.nii.gz'))
rng = random.Random(42)
synth_file_list = []

print(f'Synthesizing {N_SYNTH} tumor volumes...')

In [ ]:
def extract_patch(volume, patch_size):
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    z0, y0, x0 = max(0,(D-pd)//2), max(0,(H-ph)//2), max(0,(W-pw)//2)
    p = volume[z0:z0+pd, y0:y0+ph, x0:x0+pw]
    pad = [(0,max(0,pd-p.shape[0])),(0,max(0,ph-p.shape[1])),(0,max(0,pw-p.shape[2]))]
    p = np.pad(p, pad, constant_values=CT_WIN_MIN)
    return p[:pd, :ph, :pw]

for i in tqdm(range(N_SYNTH)):
    vol_path = rng.choice(healthy_paths)
    volume, affine = load_nifti(vol_path)
    patch = extract_patch(volume, PATCH_SIZE)

    # Generate tumor mask [PROPOSAL]
    mask = generate_tumor_mask(tuple(PATCH_SIZE), size_category='random', seed=42+i)

    # Preprocess + encode
    w = window_ct(patch, CT_WIN_MIN, CT_WIN_MAX)
    img_t = torch.from_numpy(w).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    mask_t = torch.from_numpy(mask).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    healthy_t = img_t * (1.0 - mask_t)

    with torch.no_grad():
        z_ctx = ae_model.encode_deterministic(healthy_t)
        mask_ds = F.interpolate(mask_t, size=z_ctx.shape[2:], mode='nearest')
        z_synth = diff_model.sample(mask_ds, z_ctx, num_inference_steps=N_INFER_STEPS)
        synth = ae_model.decode(z_synth).squeeze().clamp(0,1).cpu().numpy()

    synth_hu = synth * (CT_WIN_MAX - CT_WIN_MIN) + CT_WIN_MIN
    fname = f'pancreas_synth_{i:05d}'
    img_path = synth_dir / 'images' / f'{fname}.nii.gz'
    lbl_path = synth_dir / 'labels' / f'{fname}.nii.gz'
    save_nifti(synth_hu, affine, img_path)
    save_nifti(mask.astype(np.float32), affine, lbl_path)
    synth_file_list.append({'image': str(img_path), 'label': str(lbl_path)})

    if i % 10 == 0:
        show_ct_slices(synth, mask.astype(float), title=f'Synth {i}',
                       save_path=synth_dir / 'previews' / f'{fname}_preview.png')

import json
list_path = synth_dir / 'synth_filelist.json'
with open(list_path, 'w') as f:
    json.dump(synth_file_list, f, indent=2)
print(f'Done. {N_SYNTH} synthetic pairs saved.')

In [ ]:
# Visual QC — show 3 random synthetic tumors
previews = sorted((synth_dir / 'previews').glob('*.png'))
fig, axes = plt.subplots(1, min(3, len(previews)), figsize=(15, 5))
if len(previews) == 1: axes = [axes]
for ax, p in zip(axes, previews[:3]):
    from PIL import Image
    ax.imshow(Image.open(p)); ax.axis('off'); ax.set_title(p.stem)
plt.suptitle('Synthetic Tumor Previews (red = tumor mask)')
plt.tight_layout(); plt.show()

---
## Stage 3b — Segmentation Training
Train 3D U-Net on synthetic + small real data mix.  
> **Source**: Proposal Stage 3 + MONAI UNet

In [ ]:
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete

seg_cfg = CFG['segmentation']
sm = seg_cfg['model']

seg_train_loader, seg_val_loader = get_segmentation_loaders(
    raw_dir=RAW_DIR,
    organ='pancreas',
    synth_files=synth_file_list,
    patch_size=PATCH_SIZE,
    batch_size=seg_cfg['training']['batch_size'],
    num_workers=CFG['data']['num_workers'],
    hybrid_real_fraction=seg_cfg['training']['hybrid_real_fraction'],
)
print(f'Seg train: {len(seg_train_loader)} batches | val: {len(seg_val_loader)} batches')

seg_model = TumorSegmenter(
    channels=tuple(sm['channels']),
    strides=tuple(sm['strides']),
    num_res_units=sm['num_res_units'],
).to(DEVICE)
print(f'Segmenter parameters: {sum(p.numel() for p in seg_model.parameters()):,}')

In [ ]:
seg_loss_fn = get_loss()
seg_inferer = get_sliding_window_inferer(PATCH_SIZE)
seg_optimizer = torch.optim.Adam(seg_model.parameters(), lr=seg_cfg['training']['lr'])
seg_scaler = GradScaler()
SEG_MAX_EPOCHS = seg_cfg['training']['max_epochs']
SEG_VAL_INTERVAL = seg_cfg['training']['val_interval']
SEG_WARMUP = seg_cfg['training']['warmup_epochs']
seg_lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    seg_optimizer, T_max=SEG_MAX_EPOCHS - SEG_WARMUP, eta_min=1e-6
)

seg_ckpt_path = CKPT_DIR / 'segmenter_pancreas_best.pth'
best_dice = 0.0
seg_train_losses, seg_val_dices = [], []
post_pred = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)

In [ ]:
for epoch in range(1, SEG_MAX_EPOCHS + 1):
    seg_model.train()
    train_loss = 0.0
    for batch in seg_train_loader:
        x = batch['image'].to(DEVICE)
        y = batch['label'].long().to(DEVICE)
        seg_optimizer.zero_grad()
        with autocast():
            loss = seg_loss_fn(seg_model(x), y)
        seg_scaler.scale(loss).backward()
        seg_scaler.unscale_(seg_optimizer)
        torch.nn.utils.clip_grad_norm_(seg_model.parameters(), 1.0)
        seg_scaler.step(seg_optimizer)
        seg_scaler.update()
        train_loss += loss.item()
    train_loss /= len(seg_train_loader)
    seg_train_losses.append(train_loss)
    if epoch > SEG_WARMUP:
        seg_lr_scheduler.step()

    if epoch % SEG_VAL_INTERVAL == 0:
        seg_model.eval()
        dice_metric = DiceMetric(include_background=False, reduction='mean')
        with torch.no_grad():
            for batch in seg_val_loader:
                x = batch['image'].to(DEVICE)
                y = batch['label'].long().to(DEVICE)
                logits = seg_inferer(x, seg_model)
                dice_metric([post_pred(l) for l in logits], [post_label(l) for l in y])
        val_dice = dice_metric.aggregate().item()
        seg_val_dices.append((epoch, val_dice))
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save({'model': seg_model.state_dict(), 'epoch': epoch,
                        'best_dice': best_dice}, seg_ckpt_path)
        print(f'Epoch {epoch:03d}/{SEG_MAX_EPOCHS} | train_loss={train_loss:.4f} | val_dice={val_dice:.4f}'
              + (' [saved]' if val_dice == best_dice else ''))
    else:
        if epoch % 20 == 0:
            print(f'Epoch {epoch:03d}/{SEG_MAX_EPOCHS} | train_loss={train_loss:.4f}')

print(f'\nStage 3 done. Best val Dice: {best_dice:.4f}')

---
## Evaluation
Dice score, tumor-wise sensitivity (stratified by size), optional Hausdorff.
> **Source**: Proposal metrics section

In [ ]:
# Load best segmenter
ckpt = torch.load(seg_ckpt_path, map_location=DEVICE)
seg_model.load_state_dict(ckpt['model'])
seg_model.eval()

results = []
with torch.no_grad():
    for batch in tqdm(seg_val_loader, desc='Evaluating'):
        x = batch['image'].to(DEVICE)
        y_np = batch['label'][0, 0].cpu().numpy().astype(np.uint8)
        logits = seg_inferer(x, seg_model)
        pred_np = logits[0].argmax(0).cpu().numpy().astype(np.uint8)
        pred_bin = (pred_np == 1).astype(np.uint8)
        gt_bin   = (y_np == 1).astype(np.uint8)
        results.append({
            'dice': dice_score(pred_bin, gt_bin),
            'sensitivity': tumor_sensitivity(pred_bin, gt_bin),
            'strat': stratified_sensitivity(pred_bin, gt_bin),
        })

dices = [r['dice'] for r in results]
sens  = [r['sensitivity'] for r in results if not np.isnan(r['sensitivity'])]
print('=== EVALUATION RESULTS ===')
print(f'Dice:        {np.mean(dices):.4f} ± {np.std(dices):.4f}')
print(f'Sensitivity: {np.mean(sens):.4f}' if sens else 'Sensitivity: N/A')
for k in ['small','medium','large']:
    v = np.nanmean([r['strat'][k] for r in results])
    print(f'  {k:6s} tumor sensitivity: {v:.4f}')

In [ ]:
# Save results
import json
summary = {
    'dice_mean': float(np.mean(dices)), 'dice_std': float(np.std(dices)),
    'sensitivity_mean': float(np.mean(sens)) if sens else None,
    'stratified_sensitivity': {
        k: float(np.nanmean([r['strat'][k] for r in results]))
        for k in ['small','medium','large']
    },
    'n_cases': len(results),
}
results_path = Path('/content/drive/MyDrive/tumor-synthesis/outputs') / 'eval_results.json'
with open(results_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved to {results_path}')
print(json.dumps(summary, indent=2))

---
## [PROPOSAL Exp B] Low-Annotation Experiment
Re-train diffusion with only N labeled volumes (1, 5, 10, all).

In [ ]:
# Quick ablation: vary number of labeled volumes
# Set max_labeled and re-run Stage 2 + 3 for each
# (run manually for each value to save Colab time)

MAX_LABELED_OPTIONS = [5, 10, None]  # None = all
print('To run low-annotation ablation:')
print('1. Set max_labeled in get_diffusion_loaders() call')
print('2. Re-run Stage 2 training cell')
print('3. Re-run Stage 3 training + eval cells')
print('4. Record dice/sensitivity for each setting')
print(f'Options: {MAX_LABELED_OPTIONS}')